# Data Description
- Author: 詹明翰
- Date first created: 20261016（衍生檔建立日期）
- Background: 民國 114 年臺北市死傷交通事故，供樣本、光線及夜間照明三張表使用。
- Unit of observation: 原始第一當事人紀錄；暫以一列一事故。無官方事故識別碼，候選鍵唯一並不構成官方確認。
- Sample period: 2025-01-01 至 2025-12-31；依實際時間欄位驗證。
- Known issues: 原始缺值保留；座標缺失 2 筆無法判定太陽時段；光線是設備狀態，不能據此直接判定夜間。
  速限 0 保留原值，定義未確認；死亡指標僅指 24 小時內死亡，2–30 日人數另列；不能辨識重傷。
- Definition for each variable:
  - 原始 47 欄完整保留；原始定義參考 [事故對照表](../raw/事故對照表.pdf) 與 [原始資料索引](../../PROJECT.md#data-artifacts)。
  - 本次使用的原始欄位：發生年度／月／日／時／分（民國年與當地時間）、區序、肇事地點、死亡人數（24小時內）、
    2-30日死亡人數、受傷人數、當事人序號、光線、天候、道路型態、速限-速度限制（km/h）、座標-X／Y（經度／緯度）。
  - `record_id`: 原始列順序 1–22,762，僅供追溯，非官方事故編號。
  - `crash_datetime`: 民國年加 1911，Asia/Taipei 當地日期時間文字。
  - `district`: 區序移除開頭數字所得行政區。
  - `lighting_label`, `weather_label`, `road_form_label`: PDF 第 1 頁中文標籤；原始缺值保持缺值。
  - `fatal24`: 死亡人數 ≥ 1 為 1，等於 0 為 0，原始缺值為缺值。
  - `night_fixed`: 18:49（含）–05:02（不含）為 1，其餘為 0。
  - `sunrise_local`, `sunset_local`: Astral 3.2 使用該筆座標、海拔 0 公尺、UTC+8 計算，ISO 8601 文字。
  - `is_night`: 早於當日日出或晚於當日日落為 1，其餘 0；座標缺失為缺值。
- Data path: `data/processed/taipei_2025_descriptive.csv`
- Source data: [data/raw/114年-臺北市死傷交通事故明細.csv](../raw/114年-臺北市死傷交通事故明細.csv)；[PDF](../raw/事故對照表.pdf)。
- Processing code: [prepare_descriptive_statistics.ipynb](../code/prepare_descriptive_statistics.ipynb)
- Role: analysis（敘述性統計）
- Verification status: 已重新讀取並核對 22,762 列、58 欄、原始欄位保留、候選鍵唯一及日出日落缺值規則；本 Notebook 另執行下列 Assertions。
- SHA-256: `09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60`
- Downstream analysis input: `data/processed/taipei_2025_descriptive.csv`；[分析 Notebook](../../finding/敘述性統計.ipynb)。

> **小組簡報參考：** 請閱讀 [finding/敘述性統計.ipynb](../../finding/敘述性統計.ipynb)，依資料定義、樣本分布、照明與日夜交叉、夜間比較、敏感度及結論排列。本檔保留資料來源紀錄。

## 如何閱讀這份資料說明

這份 Notebook 檢查已保存的衍生資料，讓後續分析知道每個欄位代表什麼、
可以使用多少筆紀錄，以及有哪些解讀限制。它與產生資料的處理 Notebook 分工不同：
這裡只讀取資料並驗證，不重新清理或覆寫 CSV。

閱讀順序是確認資料版本、看前十筆範例、閱讀數值摘要，
再逐一閱讀七個類別變數的摘要，最後檢查 Assertions。
每個類別變數各有標題、獨立表格與結果說明，顯示全部已觀察到的類別。

## 夜間判定的計算來源與定義

**資料來源與方法。** 本研究沒有將中央氣象署公布的日出日落表匯入資料。
日出與日落時間由 Python 的 **Astral 3.2** 計算：輸入每筆事故的西元日期、
`座標-Y`（緯度）與 `座標-X`（經度），海拔統一設為 0 公尺，使用臺灣時間 UTC+8
（2025 年的 Asia/Taipei）。套件依天文模型推算太陽位置及日出日落，並非現場亮度量測。

**主要指標 `is_night`。** 令事故的當地時間為 t、該日該位置的日出與日落為 R、S：
當 **t < R 或 t > S** 時設為 1，其餘為 0；恰好等於日出或日落歸為 0。
若經度或緯度缺失，日出日落及主要夜間指標都保留缺值，不用市中心座標代替。
本資料有 2 筆這類紀錄。`光線` 是照明設備狀態，不能直接取代這個日夜判斷。

**固定指標 `night_fixed`。** 依本次確認的研究設計，採「全年最晚日落之後、
全年最早日出之前」作為保守的固定時段。先對 **2025 年完整 365 日**，
在資料全部有效座標的經緯度外接矩形四角計算日出日落，涵蓋本資料中的位置差異；
經度範圍為 121.4611803–121.6520178、緯度為 24.9659305–25.1832948。
這是本資料的空間範圍，並非臺北市官方行政邊界。

四角全年計算的最早日出為 **2025-06-09 05:02:50.800739**（東北角），
最晚日落為 **2025-07-02 18:48:12.258035**（西北角）。配合事故時間只有分鐘精度，
將最早日出向下取整為 05:02、最晚日落向上取整為 18:49。
故 `night_fixed = 1` 的時段為 **18:49（含）至翌日 05:02（不含）**，
即一天中的分鐘數 m ≥ 1129 或 m < 302，其餘設為 0。
另外逐筆驗證：所有有有效座標且被固定時段納入的事故，必須同時符合其當日 `is_night = 1`。

這個固定指標是較窄的夜間子樣本；值為 0 只表示不在固定時段內，**不代表白天**。
固定時段依時間即可套用；但缺座標者無法逐筆驗證實際位置，所以「皆為夜間」的檢查
限定於有有效座標的紀錄，不能延伸到其他年度、其他地區或未知位置。

**為何不只採夏至？** 夏至白晝最長，並不保證當天是全年日出最早或日落最晚。
此點參考 [臺北市立天文科學教育館的 2025 夏至說明](https://tam.gov.taipei/News_Content.aspx?n=B64052C7930D4913&s=C881A8681CC67490&sms=2CF1F5E2E0B96411)。
該網頁用於確認天文概念；本資料各筆時間與固定界線仍由 Astral 計算，不冒稱為官方公布值。

**解讀限制。** 日落後與日出前可能仍有暮光或曙光；天候、山勢、建築遮蔽與實際照明
也會影響亮度。因此本定義表示模型下的日出日落夜間，不能稱為「完全黑暗」。

### 套件與讀檔工具

下列程式設定專案路徑與 SHA-256 檢查，並調整表格顯示格式。
讀檔會固定使用文件指定的精確路徑；若版本或必要結構改變，程式會停止並要求先核對。

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root():
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / 'AGENTS.md').is_file() and (p / 'templates').is_dir():
            return p
    raise FileNotFoundError('請從含 AGENTS.md 與 templates/ 的專案內開啟 Notebook。')

PROJECT_ROOT = find_project_root()

def project_file(relative, area, must_exist=True):
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('請使用精確的專案相對路徑。')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('資料路徑超出預期目錄。')
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def verify(path, expected):
    actual = sha256_file(path)
    if actual != expected:
        raise ValueError(f'檔案版本改變，請核對後再執行：{path.name}')
    return actual

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## Change for Your Own Data
以固定路徑及 SHA-256 驗證實際保存的衍生檔，沒有替代檔案。

In [2]:
INPUT_RELATIVE_PATH = 'data/processed/taipei_2025_descriptive.csv'
EXPECTED_SHA256 = '09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60'
data_path = project_file(INPUT_RELATIVE_PATH, 'processed')
verify(data_path, EXPECTED_SHA256)
df = pd.read_csv(data_path, encoding='utf-8-sig')
assert df.shape == (22762, 58)
assert df['record_id'].is_unique
required = ['crash_datetime','district','lighting_label','weather_label','road_form_label',
            '死亡人數','2-30日死亡人數','受傷人數','速限-速度限制','fatal24','is_night','night_fixed']
assert set(required).issubset(df.columns)
dt = pd.to_datetime(df['crash_datetime'])

## Summary

本次輸入應為 22,762 列、58 欄。列數表示原始事故紀錄沒有因新增欄位而增減，
欄數則包含保留的 47 個原始欄位與新增的 11 個欄位。
下列雜湊供日後確認檔案版本，並不取代對欄位意義的檢查。

In [3]:
print({'rows':len(df),'columns':len(df.columns),'sha256':sha256_file(data_path)})

{'rows': 22762, 'columns': 58, 'sha256': '09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60'}


### Sample

前十筆可用來比對事故時間、照明標籤與夜間指標是否符合預期。
例如「無照明」表示設備狀態，是否屬於夜間仍須看 `is_night`；
兩個欄位回答的問題不同。這些列只是資料範例，不能用來估計全年的類別占比。

In [4]:
display(df[required].head(10))

,crash_datetime,district,lighting_label,weather_label,road_form_label,死亡人數,2-30日死亡人數,受傷人數,速限-速度限制,fatal24,is_night,night_fixed
0,2025-01-01 00:18:00,中山區,有照明且開啟,晴,直路,0.0000,0.0000,1.0000,40.0000,0.0000,1.0000,1
1,2025-01-01 00:31:00,文山區,有照明且開啟,陰,直路,0.0000,0.0000,3.0000,50.0000,0.0000,1.0000,1
2,2025-01-01 00:35:00,中山區,有照明且開啟,陰,多岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
3,2025-01-01 00:41:00,中山區,有照明且開啟,陰,直路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
4,2025-01-01 00:57:00,南港區,有照明且開啟,陰,三岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
5,2025-01-01 01:34:00,信義區,有照明且開啟,陰,三岔路,0.0000,0.0000,3.0000,30.0000,0.0000,1.0000,1
6,2025-01-01 02:15:00,信義區,有照明且開啟,陰,直路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
7,2025-01-01 06:26:00,文山區,無照明,晴,坡路,0.0000,0.0000,1.0000,30.0000,0.0000,1.0000,0
8,2025-01-01 06:29:00,文山區,有照明未開啟或故障,晴,三岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,0
9,2025-01-01 07:11:00,松山區,有照明且開啟,晴,直路,0.0000,0.0000,2.0000,30.0000,0.0000,0.0000,0


### Summary Stats

#### 數值變數：如何讀平均數與分位數

「有效 N」是該欄非缺值的紀錄數；平均數、標準差與分位數都以這些紀錄計算。
缺值數另列，並非以 0 取代。中位數與第 25／75 百分位可用來了解分布集中在哪裡，
最大值則提醒我們是否有少數較大的觀測。

死亡人數與 `fatal24` 的平均數意義不同：前者為每件平均死亡人數，
後者為死亡事故比例（0/1 指標的平均）。受傷人數為每件事故的總受傷人數，
不是第一當事人本身的傷勢。速限是道路限制，單位 km/h，不是事故當下車速。

In [5]:
def show(title, frame):
    display(Markdown('### ' + title))
    display(frame)

numeric = ['死亡人數','2-30日死亡人數','受傷人數','速限-速度限制','fatal24']
stats = df[numeric].describe(percentiles=[.25,.5,.75]).T
stats = stats.rename(columns={'count':'有效 N','mean':'平均','std':'標準差','min':'最小',
                              '25%':'第25百分位','50%':'中位數','75%':'第75百分位','max':'最大'})
stats['缺值 N'] = df[numeric].isna().sum()
stats.index = ['24小時內死亡人數','2–30日死亡人數','受傷人數','速限（km/h；保留0）','24小時內死亡事故指標（0/1）']
show(f'數值變數摘要（全樣本 N={len(df):,}）', stats)

### 數值變數摘要（全樣本 N=22,762）

,有效 N,平均,標準差,最小,第25百分位,中位數,第75百分位,最大,缺值 N
24小時內死亡人數,"22,755.0000",0.0025,0.0513,0.0000,0.0000,0.0000,0.0000,2.0000,7
2–30日死亡人數,"22,752.0000",0.0001,0.0115,0.0000,0.0000,0.0000,0.0000,1.0000,10
受傷人數,"22,761.0000",1.3269,0.5970,0.0000,1.0000,1.0000,2.0000,21.0000,1
速限（km/h；保留0）,"22,758.0000",44.1460,10.0558,0.0000,40.0000,50.0000,50.0000,80.0000,4
24小時內死亡事故指標（0/1）,"22,755.0000",0.0025,0.0500,0.0000,0.0000,0.0000,0.0000,1.0000,7


數值摘要顯示，受傷人數的中位數為 1、平均約為 1.327 人，最大值為 21 人，
表示部分事故涉及多名傷者。24 小時內死亡人數大多為 0，因此平均數很小；
不能只看平均數而忽略死亡事故件數。速限中位數為 50 km/h；原始資料中的 0 仍保留，
在未釐清其意義前，平均速限應配合原始值分布一起閱讀。

#### 類別變數：表格共同規則

每張表只摘要一個變數，百分比以該變數的全部非缺值紀錄為分母。
缺值筆數會顯示在表格前，不混入有效類別百分比；各表有效百分比加總為 100%（顯示值可能有四捨五入差異）。
月份依時間排序，其餘依件數排序；本次顯示全部類別，不截取前十類。

In [6]:
category_summaries = {}

def summarize_category(name, series, sort_by_category=False):
    valid_n = int(series.notna().sum())
    missing_n = int(series.isna().sum())
    counts = series.dropna().value_counts()
    if sort_by_category:
        counts = counts.sort_index()
    table = counts.rename_axis('類別').reset_index(name='筆數')
    table['有效樣本占比（%）'] = table['筆數'] / valid_n * 100 if valid_n else np.nan
    display(Markdown(f'**{name}**｜全樣本 N={len(series):,}；有效 N={valid_n:,}；缺值 N={missing_n:,}。'
                     f'百分比分母為 {valid_n:,} 筆有效紀錄。'))
    display(table)
    assert int(table['筆數'].sum()) == valid_n
    if valid_n:
        assert np.isclose(table['有效樣本占比（%）'].sum(), 100)
    category_summaries[name] = table
    return table

### 1. 發生月份

觀察一年內事故紀錄如何分布。月份是週期性類別，這裡計算各月件數與比例，而不計算「平均月份」。月份天數及交通量不同，件數差異不能直接解讀為事故風險差異。

In [7]:
month_summary = summarize_category('發生月份', dt.dt.month, sort_by_category=True)

**發生月份**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,1,1896,8.3297
1,2,1641,7.2094
2,3,2020,8.8744
3,4,1855,8.1495
4,5,1867,8.2023
5,6,1773,7.7893
6,7,1851,8.1320
7,8,1949,8.5625
8,9,1929,8.4747
9,10,1951,8.5713


**結果解讀：** 資料涵蓋 1 至 12 月，沒有缺月。12 月件數最多（2,063 筆），2 月最少（1,641 筆）。這是月份分布，未依每月天數或交通量標準化。

### 2. 行政區

行政區標籤由原始區序移除開頭數字而來。每列計入其記錄的行政區，百分比用來說明事故紀錄在臺北市各區的分布。

In [8]:
district_summary = summarize_category('行政區', df['district'])

**行政區**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,中山區,3015,13.2458
1,大安區,2495,10.9613
2,中正區,2174,9.5510
3,信義區,2097,9.2127
4,士林區,2074,9.1117
5,內湖區,2010,8.8305
6,萬華區,1684,7.3983
7,大同區,1639,7.2006
8,北投區,1592,6.9941
9,文山區,1504,6.6075


**結果解讀：** 12 個行政區都有紀錄，中山區最多（3,015 筆），南港區最少（1,019 筆）。區域面積、人口與交通量不同，不能只用此表判斷哪一區的道路較安全。

### 3. 照明設備狀態

依 PDF 將光線代碼轉為設備狀態。代碼 6 同時包括「未開啟」與「故障」，原始資料無法再細分，因此這裡保持原類別。此表包含全天紀錄，不代表各組都是夜間事故。

In [9]:
lighting_summary = summarize_category('照明設備狀態', df['lighting_label'])

**照明設備狀態**｜全樣本 N=22,762；有效 N=22,750；缺值 N=12。百分比分母為 22,750 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,有照明未開啟或故障,9296,40.8615
1,有照明且開啟,9058,39.8154
2,無照明,4396,19.3231


**結果解讀：** 有效紀錄為 22,750 筆，另有 12 筆缺值。「有照明未開啟或故障」9,296 筆、「有照明且開啟」9,058 筆、「無照明」4,396 筆。夜間比較還須與夜間指標交叉篩選。

### 4. 天候

天候依 PDF 對照為晴、陰、雨等文字類別。各類別百分比描述事故發生時記錄的天候，沒有使用不同天候的行車時間作為分母。

In [10]:
weather_summary = summarize_category('天候', df['weather_label'])

**天候**｜全樣本 N=22,762；有效 N=22,760；缺值 N=2。百分比分母為 22,760 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,晴,12966,56.9684
1,陰,6440,28.2953
2,雨,3353,14.7320
3,風,1,0.0044


**結果解讀：** 有效紀錄為 22,760 筆、缺值 2 筆。晴天最多（12,966 筆），其次為陰天（6,440 筆）與雨天（3,353 筆），另有 1 筆「風」。這不代表晴天較危險，也可能與晴天較常見或交通量較大有關。

### 5. 道路型態

此欄區分交岔路、直路、橋樑等型態，數字代碼沒有可直接平均的量尺意義。以下列出全部 14 個已觀察到的類別，讓較少見的型態也能被檢視。

In [11]:
road_summary = summarize_category('道路型態', df['road_form_label'])

**道路型態**｜全樣本 N=22,762；有效 N=22,750；缺值 N=12。百分比分母為 22,750 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,四岔路,8286,36.4220
1,直路,7528,33.0901
2,三岔路,4503,19.7934
3,多岔路,1278,5.6176
4,其他,389,1.7099
5,彎曲路及附近,298,1.3099
6,橋樑,242,1.0637
7,高架道路,68,0.2989
8,坡路,51,0.2242
9,圓環,41,0.1802


**結果解讀：** 有效紀錄為 22,750 筆、缺值 12 筆。四岔路（8,286 筆）與直路（7,528 筆）最多，其次為三岔路（4,503 筆）。件數較少的型態在後續分組比較時，可能只有很少的有效觀測。

### 6. 日出日落夜間指標

這是主要夜間定義：事故早於當日日出或晚於當日日落為夜間，其餘為非夜間。表格將 0／1 轉為可讀標籤；缺座標的紀錄仍是未知，不歸入非夜間。

In [12]:
solar_night_summary = summarize_category('日出日落夜間指標', df['is_night'].map({0:'非夜間', 1:'夜間'}))

**日出日落夜間指標**｜全樣本 N=22,762；有效 N=22,760；缺值 N=2。百分比分母為 22,760 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,非夜間,15965,70.1450
1,夜間,6795,29.8550


**結果解讀：** 有效紀錄為 22,760 筆，其中夜間 6,795 筆、非夜間 15,965 筆；另有 2 筆無法判定。正式夜間比較使用這 6,795 筆，再依照明設備狀態分組。

### 7. 固定時段夜間指標

採全年最晚日落後與最早日出前的保守時段：18:49（含）至翌日 05:02（不含）。界線來自資料座標範圍四角全年計算，細節見前述方法定義。套用時只需事故時間，全年使用相同界線；值為0表示不在固定時段內，不等於白天。

In [13]:
fixed_night_summary = summarize_category('固定時段夜間指標', df['night_fixed'].map({0:'固定夜間時段之外', 1:'18:49–05:02'}))

**固定時段夜間指標**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,固定夜間時段之外,17735,77.9149
1,18:49–05:02,5027,22.0851


**結果解讀：** 固定時段內有 5,027 筆，時段外 17,735 筆，沒有指標缺值。其中 0 筆缺座標而無法逐筆核對；其餘所有入選紀錄都符合當日日出日落夜間。固定時段是較窄的子樣本，不等於完整夜間樣本。

### Assertions

下列 assertions 檢查的重點包括：全部為第一當事人、候選鍵無重複、
死亡指標保留原始缺值、12 個月份完整，以及兩種夜間指標符合定義。
程式若停止，表示至少有一個已設定的條件不成立，應先查明原因；
全數通過也只代表這些特定檢查通過，不代表每項原始輸入都已被官方核實。

In [14]:
assert df['當事人序號'].eq(1).all()
candidate_key = ['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
assert not df.duplicated(candidate_key).any()
assert df['is_night'].isna().equals(df[['座標-X','座標-Y']].isna().any(axis=1))
assert df['fatal24'].isna().equals(df['死亡人數'].isna())
assert df.loc[df['死亡人數'].notna(),'fatal24'].eq(df['死亡人數'].dropna().ge(1)).all()
assert set(dt.dt.month) == set(range(1,13))
valid = df['is_night'].notna()
local_dt = pd.to_datetime(df.loc[valid,'crash_datetime']).dt.tz_localize('Asia/Taipei')
rise = pd.to_datetime(df.loc[valid,'sunrise_local'],utc=True)
setting = pd.to_datetime(df.loc[valid,'sunset_local'],utc=True)
assert df.loc[valid,'is_night'].eq((local_dt < rise) | (local_dt > setting)).all()
clock_minute=dt.dt.hour*60+dt.dt.minute
assert df['night_fixed'].eq(clock_minute.ge(1129)|clock_minute.lt(302)).all()
selected_known=df['night_fixed'].eq(1)&df['is_night'].notna()
assert df.loc[selected_known,'is_night'].eq(1).all()
verify(data_path, EXPECTED_SHA256)
print('Schema、列數、追溯流水號、候選鍵、缺值規則、12個月份與夜間指標檢查通過。')

Schema、列數、追溯流水號、候選鍵、缺值規則、12個月份與夜間指標檢查通過。


In [15]:
assert len(category_summaries) == 7
for name, table in category_summaries.items():
    assert table['類別'].is_unique, f'{name} 有重複類別'
print('七個類別變數均已分表顯示全部類別，件數與百分比分母核對通過。')

七個類別變數均已分表顯示全部類別，件數與百分比分母核對通過。


## 本次文件更新

2026-10-16：在前次補充文字與七個類別分表的基礎上，詳述 Astral 計算方法；依本次確認的全年最早日出／最晚日落規則，將 `night_fixed` 改為 18:49–05:02。衍生 CSV 已更新，舊 SHA-256 為 `9c90b54cf073cbe1fb8b5ade587f37d7f1e316ee988a05236b454413cb8b5d97`，新版為 `09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60`。其餘資料欄位與原始 CSV 未變。處理、資料說明與正式分析均重新執行，摘要與表3D已同步更新。